# CS-781 Capstone — Phase 4: Court Judgment Prediction (CJPE)

This notebook fine-tunes **your own Phase-1 language models** (A, B, C, D) to predict whether an Indian Supreme Court appeal is **accepted (1)** or **rejected (0)** from the judgment text, and writes one submission zip per experiment with a **descriptive file name**, e.g. `phase4_R1_D-GPT400M_LoRMA_last512_dev0.6512.zip`, all collected in `cs781_phase4/submissions/`.

## Before you run it
1. **Settings (right panel):** Accelerator **GPU T4 x2**, Internet **on**.
2. **Add-ons → Secrets:** turn on the secret **`HF`** (your Hugging Face token). The data is downloaded from `Exploration-Lab/CS-781-Capstone/Phase_4_CJPE`.
3. **Add Input:** your Phase-1 dataset (the same one as in Phases 2 and 3). Section 6 prints which file is used for A, B, C and D.
4. *Save Version → Save & Run All.* To continue later, attach this notebook's previous output too: finished experiments are loaded, not retrained.

## The approach
A judgment is far longer than a model can read (median ~2,200 words, up to 400,000), so each model reads **the last 512 tokens** — the court's analysis and its final order are at the end. The chunk vector (learned average over all layers, mean over the tokens) goes to a 2-way classifier. The loss is class-weighted (59% rejected / 41% accepted) and the decision threshold is tuned on dev for macro-F1. All text is lower-cased (dev and test are lower-cased, train is not). The test labels inside `test.json` are never read.

| Submission | Model | Technique | Reads |
|---|---|---|---|
| R1_D_lorma | D (GPT, 400M tokens) | **LoRMA** (compulsory) | last 512 tokens |
| H1_D_hier | D | **hierarchical**: frozen R1 embeds the last 4 chunks of 512 tokens → BiLSTM + attention | last 2,048 tokens |
| R2_B_adapter | B (GQA/RoPE) | **bottleneck adapters** | last 512 tokens |
| R4_A4_kd | A (first 4 layers) | **knowledge distillation** from R1 | last 512 tokens |
| R3_C_bitfit | C (encoder) | **BitFit** (biases + norms) | last 512 tokens |
| X1_D_lorma_first512 | D | LoRMA — ablation: the **first** 512 tokens | first 512 tokens |

**Round 2 (added after the first run):** after round 1, trained capacity mattered more than epochs, and R2 was still improving at epoch 3.

| Submission | Model | Technique | Reads |
|---|---|---|---|
| R2b_B_adapter_6ep | B | adapters, **6 epochs** (R2 had 3) | last 512 tokens |
| H3_B_hier | B | hierarchical head on the frozen R2b | last 2,048 tokens |
| H4_B_hier_devcv | B | same, head trained 5-fold on dev | last 2,048 tokens |
| R5_D_adapter | D | **adapters** (1.6M trained weights, like B), 5 epochs | last 512 tokens |
| H5_D_hier_devcv | D | hierarchical head on the frozen R5, 5-fold on dev | last 2,048 tokens |

`RUN_EXPERIMENTS` runs **7 models**, covering all four Phase-1 models: R1 (D, LoRMA), R2b + H4 (B), R5 + H5 (D), R4 (A, KD), R3 (C, BitFit). Attach the output of your round-2 run with Add Input: R2b, H4, R5 and H5 are then loaded from disk and only R1, R4 and R3 train (~1.2 h).

## Rules this follows
Only your Phase-1 models are used as starting points; no other pretrained model, weights or API. Training uses only `train.json` (dev only for model selection and the threshold). Exactly one 0/1 decision per test case — checked before every zip is written.

## 1. Setup and Hugging Face login

In [1]:
import os
import re
import sys
import json
import math
import time
import glob
import gc
import pickle
import copy
import random
import shutil
import zipfile
import hashlib
import warnings
from collections import Counter, defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)

HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF")
except Exception as e:
    HF_TOKEN = os.environ.get("HF_TOKEN")
    print("Kaggle secret 'HF' not available (" + type(e).__name__ + "); using the HF_TOKEN environment variable if set.")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    try:
        from huggingface_hub import login
        login(token=HF_TOKEN, add_to_git_credential=False)
        print("Hugging Face login successful.")
    except Exception as e:
        print("Hugging Face login failed:", repr(e))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
print("PyTorch:", torch.__version__, "| device:", DEVICE)
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))
if DEVICE.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
else:
    print("WARNING: no GPU. Use Kaggle 'GPU T4 x2' for real runs.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.
PyTorch: 2.11.0+cu128 | device: cuda
GPU 0: Tesla T4
GPU 1: Tesla T4


## 2. Configuration and experiments

Each experiment is one line: which Phase-1 model, which technique, which part of the judgment. `RUN_EXPERIMENTS` chooses what runs. **Check `PREDICTION_FORMAT` against the Phase-4 document.**

In [2]:
SEED = 42
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
PHASE4_DIR = "Phase_4_CJPE"
CLASS_NAMES = ["rejected (0)", "accepted (1)"]

INPUT_ROOT = os.environ.get("CS781_INPUT_ROOT", "/kaggle/input")
WORK_ROOT = os.environ.get("CS781_P4_WORK_ROOT",
                           "/kaggle/working/cs781_phase4" if os.path.isdir("/kaggle/working") else os.path.abspath("cs781_phase4"))
LOCAL_DATA_DIR = os.environ.get("CS781_P4_DATA_DIR", "")
SUBMISSION_DIR = os.path.join(WORK_ROOT, "submissions")
CACHE_DIR = os.path.join(WORK_ROOT, "cache")
for _d in (WORK_ROOT, SUBMISSION_DIR, CACHE_DIR):
    os.makedirs(_d, exist_ok=True)

PHASE1_MODELS = {
    "A": dict(architecture="A", pick="other"),
    "B": dict(architecture="B", pick="best"),
    "C": dict(architecture="C", pick="best"),
    "D": dict(architecture="A", pick="best"),
}
MODEL_TAGS = {"A": "A-GPT220M", "B": "B-GQA-RoPE", "C": "C-EncDec", "D": "D-GPT400M"}

DEFAULTS = dict(
    backbone="D", n_layers=None,
    technique="lorma",
    section="tail", chunk_tokens=512,
    batch_size=16, eval_batch_size=64,
    epochs=3, evals_per_epoch=2, patience=3,
    lr_backbone=3e-5, lr_peft=5e-4, lr_head=1e-3,
    weight_decay=0.01, warmup_frac=0.06, grad_clip=1.0,
    dropout=0.1, head_dropout=0.2,
    lora_r=16, lora_alpha=32, lora_scaling="standard", peft_targets="qv", lorma_variant="plus",
    adapter_dim=64, partial_top_layers=4,
    embedding_state=False, pooled_state_norm=False,
    class_weighting=True,
    teacher=None, kd_temperature=2.0, kd_weight=0.5,
    hier_base=None, hier_chunks=4, hier_hidden=128, hier_epochs=15, hier_patience=4, hier_lr=1e-3,
    hier_batch_size=64, hier_chunk_dropout=0.1,
    hier_train_on="train",
    hier_folds=5, hier_cv_epochs=8,
    seed=SEED,
)

EXPERIMENTS = {
    "R1_D_lorma":   dict(backbone="D", technique="lorma"),
    "R2_B_adapter": dict(backbone="B", technique="adapter"),
    "R3_C_bitfit":  dict(backbone="C", technique="bitfit", embedding_state=True, pooled_state_norm=True),
    "R4_A4_kd":     dict(backbone="A", n_layers=4, technique="full", lr_backbone=5e-5,
                         teacher="R1_D_lorma"),
    "H1_D_hier":    dict(backbone="D", technique="frozen", hier_base="R1_D_lorma"),
    "H2_D_hier_devcv": dict(backbone="D", technique="frozen", hier_base="R1_D_lorma",
                            hier_train_on="dev_cv"),
    "X1_D_lorma_first512": dict(backbone="D", technique="lorma", section="head"),
    "R2b_B_adapter_6ep": dict(backbone="B", technique="adapter", epochs=6),
    "H3_B_hier":        dict(backbone="B", technique="frozen", hier_base="R2b_B_adapter_6ep"),
    "H4_B_hier_devcv":  dict(backbone="B", technique="frozen", hier_base="R2b_B_adapter_6ep", hier_train_on="dev_cv"),
    "R5_D_adapter":     dict(backbone="D", technique="adapter", epochs=5),
    "H5_D_hier_devcv":  dict(backbone="D", technique="frozen", hier_base="R5_D_adapter", hier_train_on="dev_cv"),
    "X2_D_full":    dict(backbone="D", technique="full"),
    "X3_D_partial": dict(backbone="D", technique="partial"),
    "X4_D_lorma_tail1024": dict(backbone="D", technique="lorma", chunk_tokens=1024, batch_size=8),
}
RUN_EXPERIMENTS = [
    "R1_D_lorma",
    "R2b_B_adapter_6ep",
    "H4_B_hier_devcv",
    "R5_D_adapter",
    "H5_D_hier_devcv",
    "R4_A4_kd",
    "R3_C_bitfit",
]
if os.environ.get("CS781_P4_RUN"):
    RUN_EXPERIMENTS = [x for x in os.environ["CS781_P4_RUN"].split(",") if x]

PREDICTION_FORMAT = "list"
PREDICTION_KEY = "pred"

SMOKE_TEST = os.environ.get("CS781_P4_SMOKE", "0") == "1"
if SMOKE_TEST:
    DEFAULTS.update(chunk_tokens=48, batch_size=4, eval_batch_size=8, epochs=2, lora_r=4, adapter_dim=8, hier_epochs=3,
                    hier_hidden=16, hier_batch_size=8, hier_cv_epochs=3)
    for _cfg in EXPERIMENTS.values():
        if _cfg.get("n_layers"):
            _cfg["n_layers"] = 1
        if _cfg.get("chunk_tokens"):
            _cfg["chunk_tokens"] = 96
            _cfg["batch_size"] = 2
    print("*** SMOKE TEST MODE: tiny settings, results are meaningless ***")

def experiment_config(name):
    cfg = dict(DEFAULTS)
    cfg.update(EXPERIMENTS[name])
    cfg["name"] = name
    if cfg["hier_base"]:
        base = dict(DEFAULTS)
        base.update(EXPERIMENTS[cfg["hier_base"]])
        cfg.update(backbone=base["backbone"], chunk_tokens=base["chunk_tokens"], section="tail")
    return cfg

def submission_name(cfg, dev_f1):
    """e.g. phase4_R1_D-GPT400M_LoRMA_last512_dev0.6512.zip"""
    technique = {"lorma": "LoRMA", "lora": "LoRA", "adapter": "Adapter", "bitfit": "BitFit", "full": "FullFT",
                 "partial": "PartialFT", "frozen": "Frozen"}[cfg["technique"]]
    model = MODEL_TAGS[cfg["backbone"]] + (f"-first{cfg['n_layers']}layers" if cfg["n_layers"] else "")
    if cfg["teacher"]:
        technique = f"KD-from-{cfg['teacher'].split('_')[0]}"
    if cfg["hier_base"]:
        technique = f"HierBiLSTM-on-{cfg['hier_base'].split('_')[0]}" + ("-devCV" if cfg["hier_train_on"] == "dev_cv" else "")
        part = f"last{cfg['hier_chunks']}x{cfg['chunk_tokens']}"
    else:
        part = ("last" if cfg["section"] == "tail" else "first") + str(cfg["chunk_tokens"])
    return f"phase4_{cfg['name'].split('_')[0]}_{model}_{technique}_{part}_dev{dev_f1:.4f}.zip"

def restore_previous_outputs():
    """Copy a previous run of this notebook (attached with "Add Input") into the working folder, so finished
    experiments are re-used instead of trained again."""
    if not os.path.isdir(INPUT_ROOT):
        return
    copied = 0
    for dirpath, dirnames, _ in os.walk(INPUT_ROOT):
        if os.path.basename(dirpath) == "cs781_phase4":
            for sub, _, files in os.walk(dirpath):
                for name in files:
                    src = os.path.join(sub, name)
                    dst = os.path.join(WORK_ROOT, os.path.relpath(src, dirpath))
                    if not os.path.exists(dst):
                        os.makedirs(os.path.dirname(dst), exist_ok=True)
                        shutil.copy2(src, dst)
                        copied += 1
            dirnames[:] = []
        elif dirpath[len(INPUT_ROOT):].count(os.sep) >= 6:
            dirnames[:] = []
    if copied:
        print(f"Restored {copied} file(s) from a previous Phase-4 run.")

restore_previous_outputs()
print("Working folder:", WORK_ROOT)
print("Experiments to run:", RUN_EXPERIMENTS)
for _name in RUN_EXPERIMENTS:
    _c = experiment_config(_name)
    print(f"  {_name:22s} -> {submission_name(_c, 0.0).replace('_dev0.0000.zip', '_dev<score>.zip')}")

Working folder: /kaggle/working/cs781_phase4
Experiments to run: ['R1_D_lorma', 'R2b_B_adapter_6ep', 'H4_B_hier_devcv', 'R5_D_adapter', 'H5_D_hier_devcv', 'R4_A4_kd', 'R3_C_bitfit']
  R1_D_lorma             -> phase4_R1_D-GPT400M_LoRMA_last512_dev<score>.zip
  R2b_B_adapter_6ep      -> phase4_R2b_B-GQA-RoPE_Adapter_last512_dev<score>.zip
  H4_B_hier_devcv        -> phase4_H4_B-GQA-RoPE_HierBiLSTM-on-R2b-devCV_last4x512_dev<score>.zip
  R5_D_adapter           -> phase4_R5_D-GPT400M_Adapter_last512_dev<score>.zip
  H5_D_hier_devcv        -> phase4_H5_D-GPT400M_HierBiLSTM-on-R5-devCV_last4x512_dev<score>.zip
  R4_A4_kd               -> phase4_R4_A-GPT220M-first4layers_KD-from-R1_last512_dev<score>.zip
  R3_C_bitfit            -> phase4_R3_C-EncDec_BitFit_last512_dev<score>.zip


## 3. Load the CJPE data

Only the first 8,000 and last 40,000 characters of each judgment are kept in memory; labels of `test.json` are dropped on reading.

In [3]:
def load_json_file(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def write_json(obj, path):
    with open(path + ".tmp", "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)
    os.replace(path + ".tmp", path)

_HF_FILES = None

def phase4_file(name):
    """Local path of a Phase-4 file such as "train.json" (local folder, attached dataset, or Hugging Face download).
    The file may sit directly in Phase_4_CJPE/ or in a sub-folder of it."""
    global _HF_FILES
    candidates = []
    if LOCAL_DATA_DIR:
        candidates += sorted(glob.glob(os.path.join(LOCAL_DATA_DIR, "**", name), recursive=True))
    if os.path.isdir(INPUT_ROOT):
        for depth in range(0, 4):
            candidates += sorted(glob.glob(os.path.join(INPUT_ROOT, "*", *(["*"] * depth), PHASE4_DIR, "**", name), recursive=True))
    for c in candidates:
        if os.path.isfile(c):
            return c
    from huggingface_hub import hf_hub_download, list_repo_files
    if _HF_FILES is None:
        _HF_FILES = list_repo_files(HF_DATASET_REPO, repo_type="dataset", token=HF_TOKEN)
    matches = sorted((f for f in _HF_FILES if f.startswith(PHASE4_DIR + "/") and f.split("/")[-1] == name), key=len)
    if not matches:
        raise FileNotFoundError(f"{name} not found in {HF_DATASET_REPO}/{PHASE4_DIR}")
    print(f"downloading {matches[0]} ...", flush=True)
    return hf_hub_download(repo_id=HF_DATASET_REPO, filename=matches[0], repo_type="dataset", token=HF_TOKEN)

HEAD_CHARS, TAIL_CHARS = 8_000, 40_000

def clean(text):
    return " ".join(str(text or "").lower().split())

def read_cases(name, split):
    """-> list of dicts {id, head, tail, n_words, label}. For the TEST split the label is never read (test.json
    contains gold labels; using them would be cheating), and expert_* fields are ignored everywhere."""
    data = load_json_file(phase4_file(name))
    if isinstance(data, dict):
        data = [dict(v, id=k) if isinstance(v, dict) else {"id": k, "text": v} for k, v in data.items()]
    out = []
    for i, d in enumerate(tqdm(data, desc=f"reading {name}", leave=False)):
        raw = str(d.get("text") or "")
        n_words = raw.count(" ") + 1 if raw else 0
        head = clean(raw[:HEAD_CHARS])
        tail = clean(raw[-TAIL_CHARS:])
        if len(raw) > TAIL_CHARS and " " in tail:
            tail = tail.split(" ", 1)[1]
        if len(raw) > HEAD_CHARS and " " in head:
            head = head.rsplit(" ", 1)[0]
        case = {"id": str(d.get("id", i)), "head": head, "tail": tail, "n_words": n_words, "label": None}
        if split != "test":
            label = int(d["label"])
            assert label in (0, 1), (name, case["id"], d["label"])
            case["label"] = label
        out.append(case)
    del data
    gc.collect()
    return out

CASES = {"train": read_cases("train.json", "train"), "dev": read_cases("dev.json", "dev"), "test": read_cases("test.json", "test")}
_test_ids = [c["id"] for c in CASES["test"]]
assert len(set(_test_ids)) == len(_test_ids), "duplicate test ids"
LENGTH_BUCKETS = [(0, 1000, "<1k words"), (1000, 3000, "1k-3k"), (3000, 10000, "3k-10k"), (10000, 10 ** 12, ">10k")]

print(f"{'split':6s} {'cases':>7s} {'rejected':>9s} {'accepted':>9s} {'median words':>13s} {'max words':>10s}")
for _split, _cases in CASES.items():
    _w = [c["n_words"] for c in _cases]
    _y = Counter(c["label"] for c in _cases)
    print(f"{_split:6s} {len(_cases):7,} {(_y[0] if _split != 'test' else '-'):>9} {(_y[1] if _split != 'test' else '-'):>9} "
          f"{int(np.median(_w)):13,} {max(_w):10,}")
print("Document length buckets (words):", ", ".join(
    f"{lab}: train {sum(lo <= c['n_words'] < hi for c in CASES['train']):,} / dev {sum(lo <= c['n_words'] < hi for c in CASES['dev']):,}"
    for lo, hi, lab in LENGTH_BUCKETS))
print("Example (dev, last 300 characters):", CASES["dev"][0]["tail"][-300:])

downloading Phase_4_CJPE/Train/train.json ...


Phase_4_CJPE/Train/train.json: reconstructing file:   0%|          |  0.00B /  737MB            

Phase_4_CJPE/Train/train.json: downloading bytes:           |  0.00B            

reading train.json:   0%|          | 0/37387 [00:00<?, ?it/s]

downloading Phase_4_CJPE/Train/dev.json ...


Phase_4_CJPE/Train/dev.json: reconstructing file:   0%|          |  0.00B / 45.0MB            

... (3 outputs left out)

Phase_4_CJPE/Test/test.json: reconstructing file:   0%|          |  0.00B / 35.3MB            

Phase_4_CJPE/Test/test.json: downloading bytes:           |  0.00B            

reading test.json:   0%|          | 0/1517 [00:00<?, ?it/s]

split    cases  rejected  accepted  median words  max words
train   37,387     22067     15320         2,224    418,142
dev      1,988       994       994         2,622     38,301
test     1,517         -         -         2,685     36,911
Document length buckets (words): <1k words: train 8,852 / dev 150, 1k-3k: train 14,849 / dev 1,004, 3k-10k: train 12,068 / dev 760, >10k: train 1,618 / dev 74
Example (dev, last 300 characters): mself unbecoming and unworthy to hold a ny post. any sympathy or charitable view on such officials wi ll number be companyducive to keep the streams of administration pu re which is so vital for the success of our democrary. in the result we allow the appeal and set aside t he order of the tribunal.


## 4. Phase-1 tokenizer

Text is encoded exactly as in Phase 1: every word is BPE-encoded and followed by `</w>`. The tail is encoded word by word from the end.

In [4]:
SPECIAL_TOKENS = ["<pad>", "<unk>", "<s>", "</s>"]
PAD_ID = 0

class BPETokenizer:
    """Loads the byte-level BPE tokenizer saved by a Phase-1 notebook (tokenizer.json) and encodes text with the
    same merge rules. Only loading and encoding are needed here; the tokenizer is NOT retrained."""

    def __init__(self, directory):
        with open(os.path.join(directory, "tokenizer.json"), "r", encoding="utf-8") as f:
            data = json.load(f)
        self.token_to_id = {str(k): int(v) for k, v in data["token_to_id"].items()}
        self.merge_ranks = {tuple(pair): i for i, pair in enumerate(data["merges"])}
        self.unk_id = self.token_to_id["<unk>"]
        self.eow_id = self.token_to_id["</w>"]
        self.vocab_size = len(self.token_to_id)
        self._cache = {}

    def _merge(self, text):
        tokens = [str(b) for b in text.encode("utf-8")]
        while len(tokens) > 1:
            best_rank, best_index = None, -1
            for i in range(len(tokens) - 1):
                rank = self.merge_ranks.get((tokens[i], tokens[i + 1]))
                if rank is not None and (best_rank is None or rank < best_rank):
                    best_rank, best_index = rank, i
            if best_index < 0:
                break
            tokens[best_index:best_index + 2] = [tokens[best_index] + tokens[best_index + 1]]
        return [self.token_to_id.get(t, self.unk_id) for t in tokens]

    def encode_piece(self, piece):
        """Token ids of one piece of text WITHOUT the end-of-word token."""
        ids = self._cache.get(piece)
        if ids is None:
            ids = []
            for k in range(0, len(piece), 200):
                ids.extend(self._merge(piece[k:k + 200]))
            if len(self._cache) < 2_000_000:
                self._cache[piece] = ids
        return ids

def _word_ids(word, tokenizer):
    return tokenizer.encode_piece(word) + [tokenizer.eow_id]

def head_tokens(words, tokenizer, n_tokens):
    """The FIRST n_tokens tokens of a document given as a list of words."""
    ids = []
    for w in words:
        ids.extend(_word_ids(w, tokenizer))
        if len(ids) >= n_tokens:
            break
    return ids[:n_tokens] or [tokenizer.eow_id]

def tail_tokens(words, tokenizer, n_tokens):
    """The LAST n_tokens tokens of a document given as a list of words (encoded word by word from the end)."""
    pieces, total = [], 0
    for w in reversed(words):
        p = _word_ids(w, tokenizer)
        pieces.append(p)
        total += len(p)
        if total >= n_tokens:
            break
    ids = [t for p in reversed(pieces) for t in p]
    return ids[-n_tokens:] or [tokenizer.eow_id]

## 5. Metrics: macro-F1, threshold tuning, length buckets

In [5]:
def binary_scores(gold, pred):
    """gold, pred: arrays of 0/1 -> macro-F1 (mean of the F1 of both classes), per-class P/R/F1, accuracy, confusion."""
    gold, pred = np.asarray(gold, dtype=np.int64), np.asarray(pred, dtype=np.int64)
    out, f1s, ps, rs = {}, [], [], []
    for k in (0, 1):
        tp = int(((pred == k) & (gold == k)).sum())
        n_pred, n_gold = int((pred == k).sum()), int((gold == k).sum())
        p = tp / n_pred if n_pred else 0.0
        r = tp / n_gold if n_gold else 0.0
        f = 2 * p * r / (p + r) if p + r else 0.0
        out[f"class{k}"] = {"precision": p, "recall": r, "f1": f, "gold": n_gold, "predicted": n_pred}
        f1s.append(f); ps.append(p); rs.append(r)
    out.update(macro_f1=float(np.mean(f1s)), macro_precision=float(np.mean(ps)), macro_recall=float(np.mean(rs)),
               accuracy=float((gold == pred).mean()) if len(gold) else 0.0,
               confusion=[[int(((gold == a) & (pred == b)).sum()) for b in (0, 1)] for a in (0, 1)])
    return out

def best_threshold(gold, prob):
    """Decision threshold on P(accepted) that maximises dev macro-F1 (0.50 is used when nothing is better)."""
    best_t, best_f = 0.5, binary_scores(gold, prob >= 0.5)["macro_f1"]
    for t in np.arange(0.10, 0.901, 0.01):
        f = binary_scores(gold, prob >= t)["macro_f1"]
        if f > best_f + 1e-9:
            best_t, best_f = float(round(t, 2)), f
    return best_t, best_f

def scores_by_length(cases, gold, pred):
    """Macro-F1 per document-length bucket (does the model fail on long judgments, where most text is unseen?)."""
    out = {}
    for lo, hi, lab in LENGTH_BUCKETS:
        idx = [i for i, c in enumerate(cases) if lo <= c["n_words"] < hi]
        if idx:
            s = binary_scores(np.asarray(gold)[idx], np.asarray(pred)[idx])
            out[lab] = {"cases": len(idx), "macro_f1": s["macro_f1"], "accuracy": s["accuracy"]}
    return out

## 6. Phase-1 models

In [6]:
ARCH_OF_TYPE = {"decoder-only": "A", "decoder-only-gqa": "B", "encoder-decoder": "C", "efficient-local-mqa": "D-alibi"}
ARCH_DESCRIPTION = {"A": "GPT decoder", "B": "GQA/RoPE decoder", "C": "encoder-decoder", "D-alibi": "local MQA/ALiBi decoder"}

def torch_load(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")

def discover_phase1_checkpoints(root):
    """Finds the models written by the Phase-1 notebooks:
      <...>/<ARCH>/legal-model-final/{model.pt, config.json, tokenizer.json}       (normal case), or
      <...>/<ARCH>/best_model.pt + <...>/common/bpe_tokenizer_<ARCH>/tokenizer.json  (run stopped before the final save)."""
    found = []
    if not os.path.isdir(root):
        return found
    for dirpath, dirnames, filenames in os.walk(root):
        if dirpath[len(root):].count(os.sep) >= 12:
            dirnames[:] = []
        try:
            if os.path.basename(dirpath) == "legal-model-final" and "model.pt" in filenames and "tokenizer.json" in filenames:
                config = load_json_file(os.path.join(dirpath, "config.json"))
                history_path = os.path.join(dirpath, "training_history.json")
                model_file, tokenizer_dir = os.path.join(dirpath, "model.pt"), dirpath
                if os.path.exists(history_path):
                    eval_loss = float(load_json_file(history_path).get("best_eval_loss", "inf"))
                else:
                    eval_loss = float(torch_load(model_file).get("best_eval_loss", "inf"))
            elif "best_model.pt" in filenames and "legal-model-final" not in dirnames:
                common = os.path.join(os.path.dirname(dirpath), "common")
                tokenizer_dirs = [d for d in sorted(glob.glob(os.path.join(common, "bpe_tokenizer_" + os.path.basename(dirpath) + "*")))
                                  if os.path.exists(os.path.join(d, "tokenizer.json"))]
                if not tokenizer_dirs:
                    continue
                saved = torch_load(os.path.join(dirpath, "best_model.pt"))
                config, eval_loss = saved["config"], float(saved.get("eval_loss", "inf"))
                del saved
                model_file, tokenizer_dir = os.path.join(dirpath, "best_model.pt"), tokenizer_dirs[-1]
            else:
                continue
            arch = ARCH_OF_TYPE.get(config.get("type")) or str(config.get("architecture", "?"))[:1]
        except Exception as e:
            print("Skipping", dirpath, "(" + type(e).__name__ + ")")
            continue
        found.append({"arch": arch, "dir": dirpath, "model_file": model_file, "tokenizer_dir": tokenizer_dir, "config": config,
                      "eval_loss": eval_loss, "size_mb": os.path.getsize(model_file) / 1e6})
    return found

PHASE1_FOUND = discover_phase1_checkpoints(INPUT_ROOT)
PHASE1 = {}
print(f"Phase-1 checkpoints found under {INPUT_ROOT}: {len(PHASE1_FOUND)}")
for c in sorted(PHASE1_FOUND, key=lambda c: (c["arch"], c["eval_loss"])):
    print(f"  {ARCH_DESCRIPTION.get(c['arch'], c['arch']):24s} val loss {c['eval_loss']:.4f} | {c['size_mb']:.0f} MB | {c['dir']}")
_chosen = {}
for slot, spec in sorted(PHASE1_MODELS.items(), key=lambda kv: kv[1]["pick"] == "other"):
    arch, choice = spec["architecture"], spec["pick"]
    candidates = [c for c in PHASE1_FOUND if c["arch"] == arch]
    if choice == "other":
        taken = [_chosen[a]["eval_loss"] for a in _chosen if _chosen[a]["arch"] == arch]
        candidates = [c for c in candidates if all(abs(c["eval_loss"] - t) > 1e-6 for t in taken)]
        pick = max(candidates, key=lambda c: c["eval_loss"]) if candidates else None
    else:
        if choice not in ("best", "auto"):
            candidates = [c for c in candidates if choice.lower() in c["dir"].lower()]
        pick = min(candidates, key=lambda c: c["eval_loss"]) if candidates else None
    if pick is not None:
        PHASE1[slot] = _chosen[slot] = dict(pick, slot=slot)
print("Your Phase-1 models:")
for slot in PHASE1_MODELS:
    if slot in PHASE1:
        e = PHASE1[slot]
        print(f"  {slot}: {ARCH_DESCRIPTION.get(e['arch'], e['arch'])}, validation loss {e['eval_loss']:.4f} | {e['dir']}")
    else:
        print(f"  {slot}: NOT FOUND - attach the output of that Phase-1 notebook with 'Add Input'")

TOKENIZERS = {}

def tokenizer_for(slot):
    path = PHASE1[slot]["tokenizer_dir"]
    if path not in TOKENIZERS:
        TOKENIZERS[path] = BPETokenizer(path)
    return TOKENIZERS[path]

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        rms = torch.rsqrt(x.float().pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return (x * rms.to(x.dtype)) * self.weight

class AttentionA(nn.Module):
    def __init__(self, d, n_heads, dropout):
        super().__init__()
        self.n_heads, self.head_dim, self.dropout = n_heads, d // n_heads, dropout
        self.qkv = nn.Linear(d, 3 * d, bias=True)
        self.out = nn.Linear(d, d, bias=True)

    def forward(self, x):
        b, t, c = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q, k, v = [z.view(b, t, self.n_heads, self.head_dim).transpose(1, 2) for z in (q, k, v)]
        y = F.scaled_dot_product_attention(q, k, v, dropout_p=self.dropout if self.training else 0.0, is_causal=True)
        return self.out(y.transpose(1, 2).contiguous().view(b, t, c))

class BlockA(nn.Module):
    def __init__(self, d, n_heads, d_ff, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d)
        self.attn = AttentionA(d, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d)
        self.fc1 = nn.Linear(d, d_ff)
        self.fc2 = nn.Linear(d_ff, d)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = x + self.dropout(self.attn(self.ln1(x)))
        return x + self.dropout(self.fc2(F.gelu(self.fc1(self.ln2(x)))))

class BackboneA(nn.Module):
    bidirectional = False

    def __init__(self, cfg, vocab_size, n_layers, dropout):
        super().__init__()
        d = cfg["d_model"]
        self.d_model, self.max_tokens = d, cfg["context_length"]
        self.token_embedding = nn.Embedding(vocab_size, d)
        self.position_embedding = nn.Embedding(cfg["context_length"], d)
        self.blocks = nn.ModuleList([BlockA(d, cfg["n_head"], cfg["d_ff"], dropout) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d)

    def layers(self):
        return self.blocks

    def forward(self, input_ids, pad_mask=None):
        positions = torch.arange(input_ids.size(1), device=input_ids.device)
        x = self.token_embedding(input_ids) + self.position_embedding(positions)[None]
        states = []
        for block in self.blocks:
            x = block(x)
            states.append(self.ln_f(x))
        return states

def rotate_half(x):
    return torch.stack((-x[..., 1::2], x[..., ::2]), dim=-1).flatten(-2)

class AttentionB(nn.Module):
    def __init__(self, d, n_heads, n_kv_heads, max_len, dropout):
        super().__init__()
        self.q_heads, self.kv_heads, self.head_dim, self.dropout = n_heads, n_kv_heads, d // n_heads, dropout
        self.q_proj = nn.Linear(d, d, bias=False)
        self.k_proj = nn.Linear(d, n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(d, n_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(d, d, bias=False)
        inv_freq = 1.0 / (10000.0 ** (torch.arange(0, self.head_dim, 2).float() / self.head_dim))
        freqs = torch.outer(torch.arange(max_len).float(), inv_freq)
        self.register_buffer("cos", torch.cos(freqs).repeat_interleave(2, dim=-1), persistent=False)
        self.register_buffer("sin", torch.sin(freqs).repeat_interleave(2, dim=-1), persistent=False)

    def forward(self, x):
        b, t, c = x.shape
        q = self.q_proj(x).view(b, t, self.q_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(b, t, self.kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(b, t, self.kv_heads, self.head_dim).transpose(1, 2)
        cos, sin = self.cos[:t].to(q.dtype)[None, None], self.sin[:t].to(q.dtype)[None, None]
        q = q * cos + rotate_half(q) * sin
        k = k * cos + rotate_half(k) * sin
        repeat = self.q_heads // self.kv_heads
        if repeat > 1:
            k, v = k.repeat_interleave(repeat, dim=1), v.repeat_interleave(repeat, dim=1)
        y = F.scaled_dot_product_attention(q, k, v, dropout_p=self.dropout if self.training else 0.0, is_causal=True)
        return self.o_proj(y.transpose(1, 2).contiguous().view(b, t, c))

class SwiGLU(nn.Module):
    def __init__(self, d, d_ff):
        super().__init__()
        self.gate = nn.Linear(d, d_ff, bias=False)
        self.up = nn.Linear(d, d_ff, bias=False)
        self.down = nn.Linear(d_ff, d, bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))

class BlockB(nn.Module):
    def __init__(self, d, n_heads, n_kv_heads, d_ff, max_len, dropout):
        super().__init__()
        self.norm1 = RMSNorm(d)
        self.attn = AttentionB(d, n_heads, n_kv_heads, max_len, dropout)
        self.norm2 = RMSNorm(d)
        self.mlp = SwiGLU(d, d_ff)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = x + self.dropout(self.attn(self.norm1(x)))
        return x + self.dropout(self.mlp(self.norm2(x)))

class BackboneB(nn.Module):
    bidirectional = False

    def __init__(self, cfg, vocab_size, n_layers, dropout):
        super().__init__()
        d = cfg["d_model"]
        self.d_model, self.max_tokens = d, cfg["context_length"]
        self.token_embedding = nn.Embedding(vocab_size, d)
        self.blocks = nn.ModuleList([BlockB(d, cfg["n_head"], cfg["n_kv_head"], cfg["d_ff"], cfg["context_length"], dropout)
                                     for _ in range(n_layers)])
        self.norm_f = RMSNorm(d)

    def layers(self):
        return self.blocks

    def forward(self, input_ids, pad_mask=None):
        x = self.token_embedding(input_ids)
        states = []
        for block in self.blocks:
            x = block(x)
            states.append(self.norm_f(x))
        return states

class AttentionC(nn.Module):
    def __init__(self, d, n_heads, dropout):
        super().__init__()
        self.n_heads, self.head_dim, self.dropout = n_heads, d // n_heads, dropout
        self.q_proj = nn.Linear(d, d)
        self.k_proj = nn.Linear(d, d)
        self.v_proj = nn.Linear(d, d)
        self.o_proj = nn.Linear(d, d)

    def forward(self, x, attn_mask=None):
        b, t, c = x.shape
        q, k, v = [proj(x).view(b, t, self.n_heads, self.head_dim).transpose(1, 2)
                   for proj in (self.q_proj, self.k_proj, self.v_proj)]
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask, dropout_p=self.dropout if self.training else 0.0)
        return self.o_proj(y.transpose(1, 2).contiguous().view(b, t, c))

class EncoderBlockC(nn.Module):
    def __init__(self, d, n_heads, d_ff, dropout):
        super().__init__()
        self.self_attn = AttentionC(d, n_heads, dropout)
        self.ln1 = nn.LayerNorm(d)
        self.ff1 = nn.Linear(d, d_ff)
        self.ff2 = nn.Linear(d_ff, d)
        self.ln2 = nn.LayerNorm(d)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, attn_mask=None):
        x = self.ln1(x + self.dropout(self.self_attn(x, attn_mask)))
        return self.ln2(x + self.dropout(self.ff2(F.gelu(self.ff1(x)))))

class BackboneC(nn.Module):
    bidirectional = True

    def __init__(self, cfg, vocab_size, n_layers, dropout):
        super().__init__()
        d = cfg["d_model"]
        self.d_model, self.max_tokens = d, cfg["encoder_length"]
        self.token_embedding = nn.Embedding(vocab_size, d)
        self.encoder = nn.ModuleList([EncoderBlockC(d, cfg["n_head"], cfg["d_ff"], dropout) for _ in range(n_layers)])
        pe = torch.zeros(self.max_tokens, d)
        position = torch.arange(0, self.max_tokens).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d, 2).float() * (-math.log(10000.0) / d))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe, persistent=False)

    def layers(self):
        return self.encoder

    embedding_state = False

    def forward(self, input_ids, pad_mask=None):
        x = self.token_embedding(input_ids)
        states = []
        if self.embedding_state:
            scale = self.token_embedding.weight.detach().float().pow(2).mean().sqrt().clamp_min(1e-12)
            states.append(F.layer_norm(x / scale.to(x.dtype), (self.d_model,)))
        x = x + self.pe[:x.size(1)].to(x.dtype)[None]
        attn_mask = None if pad_mask is None else pad_mask[:, None, None, :]
        for block in self.encoder:
            x = block(x, attn_mask)
            states.append(x)
        return states

class AttentionD(nn.Module):
    def __init__(self, d, n_heads, dropout):
        super().__init__()
        self.n_heads, self.head_dim, self.dropout = n_heads, d // n_heads, dropout
        self.q_proj = nn.Linear(d, d, bias=False)
        self.k_proj = nn.Linear(d, self.head_dim, bias=False)
        self.v_proj = nn.Linear(d, self.head_dim, bias=False)
        self.o_proj = nn.Linear(d, d, bias=False)

    def forward(self, x, attn_bias):
        b, t, c = x.shape
        q = self.q_proj(x).view(b, t, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(b, t, 1, self.head_dim).transpose(1, 2).expand(-1, self.n_heads, -1, -1)
        v = self.v_proj(x).view(b, t, 1, self.head_dim).transpose(1, 2).expand(-1, self.n_heads, -1, -1)
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_bias.to(q.dtype),
                                           dropout_p=self.dropout if self.training else 0.0)
        return self.o_proj(y.transpose(1, 2).contiguous().view(b, t, c))

class BlockD(nn.Module):
    def __init__(self, d, n_heads, d_ff, dropout):
        super().__init__()
        self.norm1 = RMSNorm(d)
        self.attn = AttentionD(d, n_heads, dropout)
        self.norm2 = RMSNorm(d)
        self.fc1 = nn.Linear(d, d_ff)
        self.fc2 = nn.Linear(d_ff, d)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, attn_bias):
        x = x + self.dropout(self.attn(self.norm1(x), attn_bias))
        return x + self.dropout(self.fc2(F.gelu(self.fc1(self.norm2(x)))))

class BackboneD(nn.Module):
    bidirectional = False

    def __init__(self, cfg, vocab_size, n_layers, dropout):
        super().__init__()
        d = cfg["d_model"]
        self.d_model, self.max_tokens, self.window = d, cfg["context_length"], cfg["local_window"]
        self.token_embedding = nn.Embedding(vocab_size, d)
        self.blocks = nn.ModuleList([BlockD(d, cfg["n_head"], cfg["d_ff"], dropout) for _ in range(n_layers)])
        self.norm_f = RMSNorm(d)
        n = cfg["n_head"]
        self.register_buffer("slopes", 2.0 ** (-(8.0 * torch.arange(n) / n)), persistent=False)

    def layers(self):
        return self.blocks

    def attention_bias(self, t, device):
        pos = torch.arange(t, device=device, dtype=torch.float32)
        distance = pos[:, None] - pos[None, :]
        blocked = (distance < 0) | (distance > self.window - 1)
        bias = -self.slopes.to(device)[:, None, None] * distance[None]
        return bias.masked_fill(blocked[None], float("-inf"))[None]

    def forward(self, input_ids, pad_mask=None):
        x = self.token_embedding(input_ids)
        attn_bias = self.attention_bias(input_ids.size(1), input_ids.device)
        states = []
        for block in self.blocks:
            x = block(x, attn_bias)
            states.append(self.norm_f(x))
        return states

BACKBONE_CLASSES = {"A": BackboneA, "B": BackboneB, "C": BackboneC, "D-alibi": BackboneD}

@torch.no_grad()
def encoder_token_check(backbone, length=256):
    """Does the Phase-1 encoder output depend on the input tokens? Two different random token sequences are encoded;
    if the last-layer states at the same positions are almost identical, the encoder only encodes positions."""
    was_training = backbone.training
    backbone.eval()
    g = torch.Generator().manual_seed(0)
    vocab = backbone.token_embedding.weight.shape[0]
    length = min(length, backbone.max_tokens)
    ids = torch.randint(4, vocab, (2, length), generator=g)
    flag, backbone.embedding_state = backbone.embedding_state, False
    last = backbone(ids)[-1].float()
    backbone.embedding_state = flag
    same_position = F.cosine_similarity(last[0], last[1], dim=-1).mean().item()
    other_position = F.cosine_similarity(last[0], last[0].roll(1, 0), dim=-1).mean().item()
    emb_rms = backbone.token_embedding.weight.float().pow(2).mean().sqrt().item()
    pe_rms = backbone.pe[:length].float().pow(2).mean().sqrt().item()
    print(f"Encoder check: token-embedding RMS {emb_rms:.4f} vs position-encoding RMS {pe_rms:.4f} | last layer, "
          f"cosine similarity of two DIFFERENT inputs at the same position {same_position:.3f} "
          f"(neighbouring positions of one input {other_position:.3f}; close to 1.0 = the output ignores the tokens)")
    backbone.train(was_training)

def load_backbone(slot, n_layers=None, dropout=0.1, verbose=True):
    """Build the architecture and load its Phase-1 weights (optionally only the first n layers)."""
    entry = PHASE1[slot]
    arch = entry["arch"]
    cfg = entry["config"]
    state = torch_load(entry["model_file"])["model"]
    total_layers = cfg["encoder_layers"] if arch == "C" else cfg["n_layer"]
    n_layers = total_layers if n_layers is None else min(int(n_layers), total_layers)
    vocab_size = state["token_embedding.weight"].shape[0]
    backbone = BACKBONE_CLASSES[arch](cfg, vocab_size, n_layers, dropout)
    wanted = set(backbone.state_dict().keys())
    missing = wanted - set(state.keys())
    if missing:
        raise RuntimeError(f"Phase-1 checkpoint for {slot} lacks weights: {sorted(missing)[:5]} ...")
    backbone.load_state_dict({k: v for k, v in state.items() if k in wanted}, strict=True)
    if verbose and arch == "C":
        encoder_token_check(backbone)
    if verbose:
        unused = len(state) - len(wanted)
        print(f"Loaded Phase-1 model {slot} ({ARCH_DESCRIPTION.get(arch, arch)}): {n_layers}/{total_layers} layers, "
              f"{sum(p.numel() for p in backbone.parameters()) / 1e6:.2f}M parameters"
              + (f" ({unused} weight tensors not needed, e.g. removed layers or the decoder)" if unused else ""))
    return backbone

Phase-1 checkpoints found under /kaggle/input: 4
  GPT decoder              val loss 1.3086 | 258 MB | /kaggle/input/datasets/prathamk26/cs781-phase1/untitled folder/results-D/cs781_phase1/A_GPT_MHA_LearnedPos/legal-model-final
  GPT decoder              val loss 1.7840 | 258 MB | /kaggle/input/datasets/prathamk26/cs781-phase1/untitled folder/results-A/cs781_phase1/A_GPT_MHA_LearnedPos/legal-model-final
  GQA/RoPE decoder         val loss 1.6521 | 264 MB | /kaggle/input/datasets/prathamk26/cs781-phase1/untitled folder/results-B/cs781_phase1/B_GQA_RoPE_RMSNorm_SwiGLU/legal-model-final
  encoder-decoder          val loss 3.0858 | 422 MB | /kaggle/input/datasets/prathamk26/cs781-phase1/untitled folder/results-C/cs781_phase1/C_EncoderDecoder/legal-model-final
Your Phase-1 models:
  A: GPT decoder, validation loss 1.7840 | /kaggle/input/datasets/prathamk26/cs781-phase1/untitled folder/results-A/cs781_phase1/A_GPT_MHA_LearnedPos/legal-model-final
  B: GQA/RoPE decoder, validation loss 1.6521

## 7. Fine-tuning and adaptation techniques

| Technique | What is trained |
|---|---|
| `full` | every weight |
| `partial` | the top layers only |
| `bitfit` | biases and normalisation gains |
| `lorma` | low-rank multiplicative update (I + (α/r)·BA)·W0 on q/v (LoRMA+) |
| `lora` | low-rank additive update W0 + (α/r)·BA |
| `adapter` | bottleneck adapters after attention and feed-forward (+ norms) |
| `frozen` | head only |

In [7]:
class LoRALinear(nn.Module):
    """LoRA (Hu et al.): additive low-rank update,  W = W0 + (alpha / r) * B A.   B starts at zero.
    With scaling="rs" it is rsLoRA:  W = W0 + (alpha / sqrt(r)) * B A."""

    def __init__(self, base, r, alpha, rows=None, scaling="standard"):
        super().__init__()
        self.base, self.scale = base, (alpha / math.sqrt(r) if scaling == "rs" else alpha / r)
        self.rows = rows or [(0, base.out_features)]
        self.lora_A = nn.ParameterList([nn.Parameter(torch.randn(r, base.in_features) / math.sqrt(base.in_features))
                                        for _ in self.rows])
        self.lora_B = nn.ParameterList([nn.Parameter(torch.zeros(e - s, r)) for s, e in self.rows])

    def forward(self, x):
        y = self.base(x)
        parts, cursor = [], 0
        for (s, e), A, B in zip(self.rows, self.lora_A, self.lora_B):
            if s > cursor:
                parts.append(y[..., cursor:s])
            parts.append(y[..., s:e] + self.scale * F.linear(F.linear(x, A.to(x.dtype)), B.to(x.dtype)))
            cursor = e
        if cursor < y.size(-1):
            parts.append(y[..., cursor:])
        return parts[0] if len(parts) == 1 else torch.cat(parts, dim=-1)

_PERM_INDEX = {}

class LoRMALinear(nn.Module):
    """LoRMA (Low-Rank Multiplicative Adaptation): the pretrained weight is MULTIPLIED by a learned matrix.

      "plus" (LoRMA+):   W = (I + (alpha / r) * B A) W0     B starts at zero, so the multiplier starts as the identity.
      "pi"   (LoRMA-pi): W = P(B A) W0, where P shifts row i of (B A) cyclically by i positions, which makes a
                         low-rank product full rank. B's first column is ones and A[0, 0] = 1 (rest of A zero), so
                         P(B A) starts as the identity.
    Because the multiplier is on the left of W0, it acts on the layer's OUTPUT: h = W0 x, y = M h + bias.
    """

    def __init__(self, base, r, alpha, rows=None, variant="plus"):
        super().__init__()
        self.base, self.scale, self.variant = base, alpha / r, variant
        self.rows = rows or [(0, base.out_features)]
        self.lorma_A, self.lorma_B = nn.ParameterList(), nn.ParameterList()
        for s, e in self.rows:
            m = e - s
            if variant == "plus":
                A, B = torch.randn(r, m) / math.sqrt(m), torch.zeros(m, r)
            elif variant == "pi":
                A, B = torch.zeros(r, m), torch.randn(m, r) * 0.02
                A[0, 0] = 1.0
                B[:, 0] = 1.0
            else:
                raise ValueError("lorma_variant must be 'plus' or 'pi'")
            self.lorma_A.append(nn.Parameter(A))
            self.lorma_B.append(nn.Parameter(B))

    @staticmethod
    def _permuted(BA):
        m = BA.size(0)
        key = (m, BA.device)
        if key not in _PERM_INDEX:
            i = torch.arange(m, device=BA.device)
            _PERM_INDEX[key] = (i[None, :] - i[:, None]) % m
        return torch.gather(BA, 1, _PERM_INDEX[key])

    def forward(self, x):
        h = F.linear(x, self.base.weight)
        parts, cursor = [], 0
        for (s, e), A, B in zip(self.rows, self.lorma_A, self.lorma_B):
            if s > cursor:
                parts.append(h[..., cursor:s])
            hs = h[..., s:e]
            if self.variant == "plus":
                parts.append(hs + self.scale * F.linear(F.linear(hs, A.to(hs.dtype)), B.to(hs.dtype)))
            else:
                parts.append(F.linear(hs, self._permuted(B @ A).to(hs.dtype)))
            cursor = e
        if cursor < h.size(-1):
            parts.append(h[..., cursor:])
        y = parts[0] if len(parts) == 1 else torch.cat(parts, dim=-1)
        return y if self.base.bias is None else y + self.base.bias

class AdapterLinear(nn.Module):
    """Houlsby-style bottleneck adapter placed on a sub-layer's output, before the residual connection:
    y = h + up(GELU(down(h))),  h = base(x).   `up` starts at zero, so the adapter starts as the identity."""

    def __init__(self, base, bottleneck):
        super().__init__()
        self.base = base
        self.adapter_down = nn.Linear(base.out_features, bottleneck)
        self.adapter_up = nn.Linear(bottleneck, base.out_features)
        nn.init.normal_(self.adapter_down.weight, std=0.02)
        nn.init.zeros_(self.adapter_down.bias)
        nn.init.zeros_(self.adapter_up.weight)
        nn.init.zeros_(self.adapter_up.bias)

    def forward(self, x):
        h = self.base(x)
        return h + self.adapter_up(F.gelu(self.adapter_down(h)))

class IA3Linear(nn.Module):
    """IA3 (Liu et al., "(IA)^3"): learned vectors that RESCALE activations, initialised to ones (= no change).
    mode "out": rescale the layer's output (used for attention keys and values; `rows` limits it to part of a fused
    matrix).  mode "in": rescale the layer's input (used for the feed-forward hidden activations, before fc2)."""

    def __init__(self, base, mode, rows=None):
        super().__init__()
        self.base, self.mode = base, mode
        if mode == "in":
            self.ia3_scale = nn.Parameter(torch.ones(base.in_features))
            self.rows = None
        else:
            self.rows = rows or [(0, base.out_features)]
            index = torch.cat([torch.arange(s, e) for s, e in self.rows])
            self.register_buffer("row_index", index, persistent=False)
            self.ia3_scale = nn.Parameter(torch.ones(len(index)))

    def forward(self, x):
        if self.mode == "in":
            return self.base(x * self.ia3_scale.to(x.dtype))
        y = self.base(x)
        full = torch.ones(y.size(-1), device=y.device, dtype=y.dtype).scatter(0, self.row_index, self.ia3_scale.to(y.dtype))
        return y * full

QV_TARGETS = {"A": ["attn.qkv"], "B": ["attn.q_proj", "attn.v_proj"], "C": ["self_attn.q_proj", "self_attn.v_proj"],
              "D-alibi": ["attn.q_proj", "attn.v_proj"]}
ALL_TARGETS = {"A": ["attn.qkv", "attn.out", "fc1", "fc2"],
               "B": ["attn.q_proj", "attn.k_proj", "attn.v_proj", "attn.o_proj", "mlp.gate", "mlp.up", "mlp.down"],
               "C": ["self_attn.q_proj", "self_attn.k_proj", "self_attn.v_proj", "self_attn.o_proj", "ff1", "ff2"],
               "D-alibi": ["attn.q_proj", "attn.k_proj", "attn.v_proj", "attn.o_proj", "fc1", "fc2"]}
SUBLAYER_OUTPUTS = {"A": ["attn.out", "fc2"], "B": ["attn.o_proj", "mlp.down"], "C": ["self_attn.o_proj", "ff2"],
                    "D-alibi": ["attn.o_proj", "fc2"]}
IA3_TARGETS = {"A": [("attn.qkv", "out"), ("fc2", "in")], "B": [("attn.k_proj", "out"), ("attn.v_proj", "out"), ("mlp.down", "in")],
               "C": [("self_attn.k_proj", "out"), ("self_attn.v_proj", "out"), ("ff2", "in")],
               "D-alibi": [("attn.k_proj", "out"), ("attn.v_proj", "out"), ("fc2", "in")]}
NEW_WEIGHT_MARKERS = ("lora_", "lorma_", "adapter_", "ia3_")

def _get(module, path):
    for part in path.split("."):
        module = getattr(module, part)
    return module

def _set(module, path, value):
    parts = path.split(".")
    for part in parts[:-1]:
        module = getattr(module, part)
    setattr(module, parts[-1], value)

def apply_technique(backbone, arch, cfg):
    """Freeze the pretrained model, then make trainable exactly what the chosen technique allows."""
    technique = cfg["technique"]
    for p in backbone.parameters():
        p.requires_grad = False
    blocks = backbone.layers()
    norm_types = (nn.LayerNorm, RMSNorm)
    if technique == "full":
        for p in backbone.parameters():
            p.requires_grad = True
    elif technique == "partial":
        for block in list(blocks)[-int(cfg["partial_top_layers"]):]:
            for p in block.parameters():
                p.requires_grad = True
        for name in ("ln_f", "norm_f"):
            if hasattr(backbone, name):
                for p in getattr(backbone, name).parameters():
                    p.requires_grad = True
    elif technique == "bitfit":
        for name, p in backbone.named_parameters():
            if name.endswith(".bias"):
                p.requires_grad = True
        for module in backbone.modules():
            if isinstance(module, norm_types):
                for p in module.parameters():
                    p.requires_grad = True
    elif technique in ("lora", "lorma"):
        targets = (QV_TARGETS if cfg["peft_targets"] == "qv" else ALL_TARGETS)[arch]
        for block in blocks:
            for path in targets:
                base = _get(block, path)
                rows = None
                if arch == "A" and path == "attn.qkv" and cfg["peft_targets"] == "qv":
                    d = base.in_features
                    rows = [(0, d), (2 * d, 3 * d)]
                if technique == "lora":
                    wrapped = LoRALinear(base, cfg["lora_r"], cfg["lora_alpha"], rows, cfg.get("lora_scaling", "standard"))
                else:
                    wrapped = LoRMALinear(base, cfg["lora_r"], cfg["lora_alpha"], rows, cfg["lorma_variant"])
                _set(block, path, wrapped)
    elif technique == "ia3":
        for block in blocks:
            for path, mode in IA3_TARGETS[arch]:
                base = _get(block, path)
                rows = None
                if arch == "A" and path == "attn.qkv":
                    d = base.in_features
                    rows = [(d, 2 * d), (2 * d, 3 * d)]
                _set(block, path, IA3Linear(base, mode, rows))
    elif technique == "adapter":
        for block in blocks:
            for path in SUBLAYER_OUTPUTS[arch]:
                _set(block, path, AdapterLinear(_get(block, path), cfg["adapter_dim"]))
        for module in backbone.modules():
            if isinstance(module, norm_types):
                for p in module.parameters():
                    p.requires_grad = True
    elif technique != "frozen":
        raise ValueError("Unknown technique: " + str(technique))
    for name, p in backbone.named_parameters():
        if any(marker in name for marker in NEW_WEIGHT_MARKERS):
            p.requires_grad = True
    return backbone

## 8. Models: chunk classifier and hierarchical head

In [8]:
class CJPEClassifier(nn.Module):
    """Phase-1 language model + a 2-way decision head.

    1. The language model reads one chunk of the judgment (e.g. its last 512 tokens).
    2. Chunk vector: a learned weighted average of ALL layers ("scalar mix"), averaged over the chunk's tokens.
    3. LayerNorm -> dropout -> linear layer -> scores for rejected (0) / accepted (1).
    """

    def __init__(self, backbone, cfg):
        super().__init__()
        self.backbone = backbone
        d = backbone.d_model
        has_embedding_state = bool(getattr(backbone, "embedding_state", False))
        self.mix = nn.Parameter(torch.zeros(len(backbone.layers()) + int(has_embedding_state)))
        self.pool_norm = bool(cfg.get("pooled_state_norm", False))
        if self.pool_norm and has_embedding_state:
            with torch.no_grad():
                self.mix[0] = math.log(max(1, len(backbone.layers())))
        self.norm = nn.LayerNorm(d)
        self.drop = nn.Dropout(cfg["head_dropout"])
        self.out = nn.Linear(d, 2)

    def chunk_vectors(self, input_ids, pad_mask):
        """[N, T] token ids -> [N, d] float32 chunk vectors (after the LayerNorm; used by the hierarchical model)."""
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            states = self.backbone(input_ids, pad_mask)
        weights = torch.softmax(self.mix, dim=0)
        assert len(states) == len(weights), (len(states), len(weights))
        m = pad_mask[:, :, None].float()
        if self.pool_norm:
            pooled = [F.layer_norm((s.float() * m).sum(1) / m.sum(1).clamp(min=1.0), (s.size(-1),)) for s in states]
            v = sum(w * p for w, p in zip(weights, pooled))
        else:
            h = sum(w * s.float() for w, s in zip(weights, states))
            v = (h * m).sum(1) / m.sum(1).clamp(min=1.0)
        return self.norm(v)

    def forward(self, input_ids, pad_mask):
        return self.out(self.drop(self.chunk_vectors(input_ids, pad_mask)))

def classification_loss(logits, labels, cfg, class_weights=None, teacher=None):
    """Cross-entropy (class-weighted), plus knowledge distillation from the teacher's softened scores if given."""
    hard = F.cross_entropy(logits, labels, weight=class_weights)
    if teacher is None:
        return hard, hard.detach(), None
    T, w = float(cfg["kd_temperature"]), float(cfg["kd_weight"])
    soft = F.kl_div(F.log_softmax(logits / T, dim=-1), F.softmax(teacher / T, dim=-1), reduction="batchmean") * T * T
    return (1 - w) * hard + w * soft, hard.detach(), soft.detach()

class HierarchicalHead(nn.Module):
    """Long-context model on top of a FROZEN fine-tuned chunk model (as in the ILDC paper's hierarchical models).
    The last K chunks of the judgment are embedded once by the chunk model; a BiLSTM reads the sequence of chunk
    vectors and attention pooling decides which chunks matter for the decision."""

    def __init__(self, d, cfg):
        super().__init__()
        self.drop = nn.Dropout(cfg["head_dropout"])
        self.lstm = nn.LSTM(d, cfg["hier_hidden"], batch_first=True, bidirectional=True)
        h = 2 * cfg["hier_hidden"]
        self.attn = nn.Sequential(nn.Linear(h, h), nn.Tanh(), nn.Linear(h, 1))
        self.out = nn.Linear(h, 2)

    def forward(self, vectors, lengths, return_attention=False):
        """vectors [B, K, d] (chunks in document order, padded at the end), lengths [B] -> scores [B, 2]."""
        x = self.drop(vectors)
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        x, _ = nn.utils.rnn.pad_packed_sequence(self.lstm(packed)[0], batch_first=True, total_length=vectors.size(1))
        mask = torch.arange(vectors.size(1), device=vectors.device)[None] < lengths.to(vectors.device)[:, None]
        a = self.attn(x).squeeze(-1).masked_fill(~mask, float("-inf"))
        a = torch.softmax(a, dim=1)
        pooled = (a[:, :, None] * x).sum(1)
        logits = self.out(self.drop(pooled))
        return (logits, a) if return_attention else logits

## 9. Training and prediction

Step-based training with two dev evaluations per epoch; the best evaluation is kept, and a run interrupted mid-epoch resumes from its last evaluation.

In [9]:
TOKENS = {}

def _tail_tokens_needed(tokenizer_dir):
    """Longest tail any experiment in RUN_EXPERIMENTS needs from this tokenizer (encoded once, then sliced)."""
    need = 0
    for name in RUN_EXPERIMENTS:
        if name not in EXPERIMENTS:
            continue
        c = experiment_config(name)
        if c["backbone"] in PHASE1 and PHASE1[c["backbone"]]["tokenizer_dir"] == tokenizer_dir and c["section"] == "tail":
            need = max(need, c["chunk_tokens"] * (c["hier_chunks"] if c["hier_base"] else 1))
    return need

def document_tokens(slot, section, n_tokens):
    """{split: [np.int32 token ids of each case]} - the first (head) or last (tail) n_tokens tokens of every judgment."""
    tokenizer_dir = PHASE1[slot]["tokenizer_dir"]
    if section == "tail":
        n_encode = max(n_tokens, _tail_tokens_needed(tokenizer_dir))
    else:
        n_encode = n_tokens
    sizes = "-".join(str(len(CASES[s])) for s in ("train", "dev", "test"))
    key = hashlib.md5(f"{tokenizer_dir}|{section}|{n_encode}|{sizes}|{HEAD_CHARS}|{TAIL_CHARS}".encode()).hexdigest()[:12]
    if key not in TOKENS:
        path = os.path.join(CACHE_DIR, f"tokens_{section}{n_encode}_{key}.pkl")
        if os.path.exists(path):
            with open(path, "rb") as f:
                TOKENS[key] = pickle.load(f)
            print(f"tokenised {section} {n_encode} tokens for model {slot}: loaded from {path}")
        else:
            tokenizer = tokenizer_for(slot)
            out = {}
            for split in ("train", "dev", "test"):
                encode = head_tokens if section == "head" else tail_tokens
                out[split] = [np.asarray(encode(c[section].split(" "), tokenizer, n_encode), dtype=np.int32)
                              for c in tqdm(CASES[split], desc=f"tokenising {split} ({section} {n_encode}) for model {slot}")]
            with open(path + ".tmp", "wb") as f:
                pickle.dump(out, f, protocol=4)
            os.replace(path + ".tmp", path)
            TOKENS[key] = out
            lengths = [len(x) for x in out["train"]]
            print(f"tokenised {section} {n_encode} tokens for model {slot}: train cases shorter than that: "
                  f"{sum(l < n_encode for l in lengths):,} / {len(lengths):,}")
    full = TOKENS[key]
    if n_encode == n_tokens:
        return full
    return {split: [x[-n_tokens:] for x in seqs] for split, seqs in full.items()}

def token_batch(seqs):
    """List of token-id arrays -> padded ids [N, T] (long) and mask [N, T] on the device."""
    width = max(len(s) for s in seqs)
    ids = np.full((len(seqs), width), PAD_ID, dtype=np.int64)
    mask = np.zeros((len(seqs), width), dtype=bool)
    for i, s in enumerate(seqs):
        ids[i, :len(s)], mask[i, :len(s)] = s, True
    return torch.from_numpy(ids).to(DEVICE), torch.from_numpy(mask).to(DEVICE)

def labels_of(split):
    return np.asarray([c["label"] for c in CASES[split]], dtype=np.int64)

@torch.no_grad()
def predict_logits(model, seqs, batch_size, desc="predict", vectors=False):
    """Scores [N, 2] (or chunk vectors [N, d]) for a list of token sequences, in their original order."""
    model.eval()
    order = sorted(range(len(seqs)), key=lambda i: len(seqs[i]))
    out = None
    for i in tqdm(range(0, len(order), batch_size), desc=desc, leave=False):
        idx = order[i:i + batch_size]
        ids, mask = token_batch([seqs[j] for j in idx])
        y = (model.chunk_vectors(ids, mask) if vectors else model(ids, mask)).float().cpu().numpy()
        if out is None:
            out = np.zeros((len(seqs), y.shape[1]), dtype=np.float32)
        out[idx] = y
    return out

def probabilities(logits):
    z = logits - logits.max(1, keepdims=True)
    e = np.exp(z)
    return (e / e.sum(1, keepdims=True))[:, 1]

def dev_report(prob, gold):
    """Dev metrics at the threshold that maximises macro-F1, and at 0.5."""
    t, _ = best_threshold(gold, prob)
    m = binary_scores(gold, prob >= t)
    m.update(threshold=t, macro_f1_at_0_5=binary_scores(gold, prob >= 0.5)["macro_f1"])
    return m

def exp_dir(name):
    path = os.path.join(WORK_ROOT, name)
    os.makedirs(path, exist_ok=True)
    return path

def build_model(cfg, verbose=True):
    backbone = load_backbone(cfg["backbone"], cfg["n_layers"], cfg["dropout"], verbose=verbose)
    if cfg["chunk_tokens"] > backbone.max_tokens:
        raise ValueError(f"chunk_tokens {cfg['chunk_tokens']} > the model's limit {backbone.max_tokens}")
    apply_technique(backbone, PHASE1[cfg["backbone"]]["arch"], cfg)
    if cfg.get("embedding_state"):
        if not hasattr(backbone, "embedding_state"):
            raise ValueError("embedding_state is only implemented for the encoder (model C)")
        backbone.embedding_state = True
    return CJPEClassifier(backbone, cfg).to(DEVICE)

def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    backbone = sum(p.numel() for p in model.backbone.parameters()) if hasattr(model, "backbone") else 0
    backbone_trainable = sum(p.numel() for p in model.backbone.parameters() if p.requires_grad) if backbone else 0
    return {"total": total, "trainable": trainable, "trainable_percent": 100.0 * trainable / max(1, total),
            "language_model": backbone, "language_model_trainable": backbone_trainable, "head": total - backbone}

def trainable_state(model):
    return {name: p.detach().cpu().clone() for name, p in model.named_parameters() if p.requires_grad}

def load_trainable(model, state):
    result = model.load_state_dict(state, strict=False)
    if result.unexpected_keys:
        raise RuntimeError("Saved weights do not fit this model: " + str(result.unexpected_keys[:5]))

def save_atomic(obj, path):
    torch.save(obj, path + ".tmp")
    os.replace(path + ".tmp", path)

def config_signature(cfg, extra=""):
    entry = PHASE1[cfg["backbone"]]
    payload = {"cfg": {k: v for k, v in cfg.items() if k != "name"}, "phase1": [entry["config"], entry["eval_loss"]],
               "data": [len(CASES[s]) for s in ("train", "dev", "test")], "extra": extra}
    return hashlib.md5(json.dumps(payload, sort_keys=True, default=str).encode()).hexdigest()

def optimizer_groups(model, cfg):
    backbone_lr = cfg["lr_peft"] if cfg["technique"] in ("bitfit", "adapter", "ia3") else cfg["lr_backbone"]
    buckets = defaultdict(list)
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if not name.startswith("backbone."):
            lr = cfg["lr_head"]
        elif any(marker in name for marker in NEW_WEIGHT_MARKERS):
            lr = cfg["lr_peft"]
        else:
            lr = backbone_lr
        decay = cfg["weight_decay"] if p.ndim >= 2 else 0.0
        buckets[(lr, decay)].append(p)
    return [{"params": params, "lr": lr, "base_lr": lr, "weight_decay": decay} for (lr, decay), params in buckets.items()]

def lr_factor(step, total_steps, warmup_frac):
    warmup = max(1, int(total_steps * warmup_frac))
    if step < warmup:
        return (step + 1) / warmup
    progress = min(1.0, (step - warmup) / max(1, total_steps - warmup))
    return 0.05 + 0.95 * 0.5 * (1.0 + math.cos(math.pi * progress))

def release(*objects):
    for obj in objects:
        del obj
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

def class_weight_tensor(labels):
    counts = np.bincount(labels, minlength=2).astype(np.float64)
    w = counts.sum() / (2.0 * np.maximum(counts, 1.0))
    return torch.tensor(w, dtype=torch.float32, device=DEVICE)

def train_classifier(cfg, signature, teacher_logits=None):
    """Train on train.json, evaluate on dev evals_per_epoch times per epoch, keep the evaluation with the best
    dev macro-F1 (threshold tuned). Resumes from the last evaluation if the session was interrupted."""
    folder = exp_dir(cfg["name"])
    last_path, model_path = os.path.join(folder, "last.pt"), os.path.join(folder, "model.pt")
    torch.manual_seed(cfg["seed"])
    model = build_model(cfg)
    params = count_parameters(model)
    print(f"[{cfg['name']}] parameters: {params['total'] / 1e6:.2f}M total | trained: "
          f"{params['language_model_trainable'] / 1e6:.3f}M of the {params['language_model'] / 1e6:.2f}M language-model weights "
          f"+ {params['head'] / 1e6:.4f}M head ({params['trainable_percent']:.2f}% of all weights)")
    tokens = document_tokens(cfg["backbone"], cfg["section"], cfg["chunk_tokens"])
    train_seqs, dev_seqs = tokens["train"], tokens["dev"]
    y_train, y_dev = labels_of("train"), labels_of("dev")
    size = cfg["batch_size"]
    steps_per_epoch = math.ceil(len(train_seqs) / size)
    total_steps = steps_per_epoch * cfg["epochs"]
    eval_every = max(1, steps_per_epoch // max(1, cfg["evals_per_epoch"]))
    optimizer = torch.optim.AdamW(optimizer_groups(model, cfg), betas=(0.9, 0.98))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    class_weights = class_weight_tensor(y_train) if cfg["class_weighting"] else None

    state = {"step": 0, "history": [], "best_f1": -1.0, "best_step": 0, "best_threshold": 0.5, "bad_evals": 0,
             "seconds": 0.0, "peak_memory_mb": 0.0, "loss_sum": 0.0, "n_loss": 0}
    if os.path.exists(last_path):
        saved = torch_load(last_path)
        if saved.get("signature") == signature:
            load_trainable(model, saved["model"])
            optimizer.load_state_dict(saved["optimizer"])
            scaler.load_state_dict(saved["scaler"])
            state = saved["state"]
            print(f"[{cfg['name']}] resuming after step {state['step']:,} of {total_steps:,}")
        del saved
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()

    def evaluate_now(epoch_float):
        t_eval = time.time()
        prob = probabilities(predict_logits(model, dev_seqs, cfg["eval_batch_size"], desc="dev evaluation"))
        m = dev_report(prob, y_dev)
        record = {"step": state["step"], "epoch": round(epoch_float, 2),
                  "train_loss": state["loss_sum"] / max(1, state["n_loss"]),
                  "dev_macro_f1": m["macro_f1"], "dev_threshold": m["threshold"], "dev_macro_f1_at_0_5": m["macro_f1_at_0_5"],
                  "dev_accuracy": m["accuracy"]}
        state["loss_sum"], state["n_loss"] = 0.0, 0
        message = (f"[{cfg['name']}] epoch {epoch_float:4.2f} | train loss {record['train_loss']:.4f} | dev macro-F1 "
                   f"{m['macro_f1']:.4f} (threshold {m['threshold']:.2f}; at 0.50: {m['macro_f1_at_0_5']:.4f}) | "
                   f"accuracy {m['accuracy']:.4f}")
        if m["macro_f1"] > state["best_f1"]:
            state.update(best_f1=m["macro_f1"], best_step=state["step"], best_threshold=m["threshold"], bad_evals=0)
            save_atomic({"model": trainable_state(model), "signature": signature, "step": state["step"],
                         "threshold": m["threshold"]}, model_path)
            message += "  <- best"
        else:
            state["bad_evals"] += 1
        state["history"].append(record)
        if DEVICE.type == "cuda":
            state["peak_memory_mb"] = max(state["peak_memory_mb"], torch.cuda.max_memory_allocated() / 1e6)
        save_atomic({"model": trainable_state(model), "optimizer": optimizer.state_dict(), "scaler": scaler.state_dict(),
                     "state": state, "signature": signature}, last_path)
        print(message + f" | eval {time.time() - t_eval:.0f} s", flush=True)
        model.train()

    stopped_early = state["bad_evals"] >= cfg["patience"]
    while state["step"] < total_steps and not stopped_early:
        epoch = state["step"] // steps_per_epoch
        first = state["step"] - epoch * steps_per_epoch
        order = np.random.RandomState(cfg["seed"] * 1000 + epoch).permutation(len(train_seqs))
        torch.manual_seed(cfg["seed"] * 1000 + epoch + first)
        model.train()
        t0 = time.time()
        bar = tqdm(range(first, steps_per_epoch), desc=f"{cfg['name']} epoch {epoch + 1}/{cfg['epochs']}", leave=False)
        for b in bar:
            idx = order[b * size:(b + 1) * size]
            for group in optimizer.param_groups:
                group["lr"] = group["base_lr"] * lr_factor(state["step"], total_steps, cfg["warmup_frac"])
            ids, mask = token_batch([train_seqs[i] for i in idx])
            logits = model(ids, mask).float()
            teacher = torch.from_numpy(teacher_logits[idx]).to(DEVICE) if teacher_logits is not None else None
            loss, _, _ = classification_loss(logits, torch.from_numpy(y_train[idx]).to(DEVICE), cfg, class_weights, teacher)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], cfg["grad_clip"])
            scaler.step(optimizer)
            scaler.update()
            state["step"] += 1
            state["loss_sum"] += loss.item()
            state["n_loss"] += 1
            if b % 20 == 0:
                bar.set_postfix(loss=f"{state['loss_sum'] / max(1, state['n_loss']):.4f}")
            at_epoch_end = b == steps_per_epoch - 1
            if (b + 1) % eval_every == 0 and (steps_per_epoch - (b + 1)) >= eval_every // 2 or at_epoch_end:
                state["seconds"] += time.time() - t0
                t0 = time.time()
                evaluate_now(state["step"] / steps_per_epoch)
                if state["bad_evals"] >= cfg["patience"]:
                    stopped_early = True
                    break
        bar.close()
    if stopped_early:
        print(f"[{cfg['name']}] early stop: no improvement in {cfg['patience']} evaluations.")
    saved = torch_load(model_path)
    load_trainable(model, saved["model"])
    if os.path.exists(last_path):
        os.remove(last_path)
    info = {"best_step": state["best_step"], "best_epoch": round(state["best_step"] / steps_per_epoch, 2),
            "history": state["history"], "threshold": saved["threshold"], "train_seconds": state["seconds"],
            "peak_memory_mb": state["peak_memory_mb"], "parameters": params, "train_cases": len(train_seqs)}
    return model, info

def load_classifier(cfg):
    model = build_model(cfg, verbose=False)
    saved = torch_load(os.path.join(exp_dir(cfg["name"]), "model.pt"))
    load_trainable(model, saved["model"])
    return model, saved["threshold"]

def chunk_sequences(seqs, n_chunks, chunk_tokens, min_tokens=32):
    """Cut the tail of every case (already the last n_chunks * chunk_tokens tokens) into chunks of chunk_tokens,
    counted from the END, kept in document order. A very short first piece is dropped unless it is the only one."""
    out = []
    for s in seqs:
        pieces = []
        end = len(s)
        while end > 0 and len(pieces) < n_chunks:
            start = max(0, end - chunk_tokens)
            pieces.append(s[start:end])
            end = start
        pieces = pieces[::-1]
        if len(pieces) > 1 and len(pieces[0]) < min_tokens:
            pieces = pieces[1:]
        out.append(pieces)
    return out

def hierarchical_vectors(cfg, base_cfg, base_signature):
    """{split: [float16 array [k, d] of chunk vectors]} from the frozen base model; cached on disk."""
    path = os.path.join(CACHE_DIR, f"chunkvectors_{cfg['hier_base']}_{cfg['hier_chunks']}x{base_cfg['chunk_tokens']}_{base_signature[:10]}.pt")
    if os.path.exists(path):
        print(f"[{cfg['name']}] chunk vectors loaded from {path}")
        return torch_load(path)
    model, _ = load_classifier(base_cfg)
    tokens = document_tokens(base_cfg["backbone"], "tail", base_cfg["chunk_tokens"] * cfg["hier_chunks"])
    out = {}
    for split in ("train", "dev", "test"):
        chunks = chunk_sequences(tokens[split], cfg["hier_chunks"], base_cfg["chunk_tokens"])
        flat = [c for pieces in chunks for c in pieces]
        print(f"[{cfg['name']}] {split}: embedding {len(flat):,} chunks of up to {base_cfg['chunk_tokens']} tokens "
              f"({len(flat) / max(1, len(chunks)):.2f} per case) with the frozen {cfg['hier_base']} model", flush=True)
        vec = predict_logits(model, flat, base_cfg["eval_batch_size"], desc=f"chunk vectors {split}", vectors=True)
        vec = vec.astype(np.float16)
        pos, per_case = 0, []
        for pieces in chunks:
            per_case.append(vec[pos:pos + len(pieces)])
            pos += len(pieces)
        out[split] = per_case
    release(model)
    save_atomic(out, path)
    return out

def hier_batch(vectors, idx, train=False, drop=0.0, rng=None):
    rows = []
    for i in idx:
        v = vectors[i]
        if train and drop > 0 and len(v) > 1:
            keep = rng.random_sample(len(v)) >= drop
            if not keep.any():
                keep[rng.randint(len(v))] = True
            v = v[keep]
        rows.append(v)
    width = max(len(v) for v in rows)
    x = np.zeros((len(rows), width, rows[0].shape[1]), dtype=np.float32)
    for r, v in enumerate(rows):
        x[r, :len(v)] = v
    return torch.from_numpy(x).to(DEVICE), torch.tensor([len(v) for v in rows])

@torch.no_grad()
def hier_logits(head, vectors, batch_size=256):
    head.eval()
    out = []
    for i in range(0, len(vectors), batch_size):
        x, lengths = hier_batch(vectors, range(i, min(len(vectors), i + batch_size)))
        out.append(head(x, lengths).float().cpu().numpy())
    return np.concatenate(out)

def train_hierarchical(cfg, train_vectors, y_train, val_vectors, y_val, signature, model_path, fixed_epochs=None, tag=""):
    """Train the BiLSTM + attention head. With validation data: keep the epoch with the best macro-F1 (early stopping).
    With fixed_epochs (cross-validation folds): train exactly that many epochs, no peeking at held-out cases."""
    torch.manual_seed(cfg["seed"])
    head = HierarchicalHead(train_vectors[0].shape[1], cfg).to(DEVICE)
    params = count_parameters(head)
    optimizer = torch.optim.AdamW(head.parameters(), lr=cfg["hier_lr"], weight_decay=cfg["weight_decay"])
    class_weights = class_weight_tensor(y_train) if cfg["class_weighting"] else None
    rng = np.random.RandomState(cfg["seed"])
    best, bad, history, t_start = -1.0, 0, [], time.time()
    size = cfg["hier_batch_size"]
    n_epochs = fixed_epochs or cfg["hier_epochs"]
    steps = math.ceil(len(y_train) / size) * n_epochs
    step = 0
    for epoch in range(n_epochs):
        head.train()
        order = rng.permutation(len(y_train))
        loss_sum, n = 0.0, 0
        for i in tqdm(range(0, len(order), size), desc=f"{cfg['name']}{tag} epoch {epoch + 1}/{n_epochs}", leave=False):
            idx = order[i:i + size]
            for g in optimizer.param_groups:
                g["lr"] = cfg["hier_lr"] * lr_factor(step, steps, cfg["warmup_frac"])
            x, lengths = hier_batch(train_vectors, idx, train=True, drop=cfg["hier_chunk_dropout"], rng=rng)
            loss = F.cross_entropy(head(x, lengths).float(), torch.from_numpy(y_train[idx]).to(DEVICE), weight=class_weights)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(head.parameters(), cfg["grad_clip"])
            optimizer.step()
            step += 1
            loss_sum += loss.item()
            n += 1
        record = {"epoch": epoch + 1, "train_loss": loss_sum / max(1, n)}
        if fixed_epochs:
            history.append(record)
            continue
        m = dev_report(probabilities(hier_logits(head, val_vectors)), y_val)
        record.update(dev_macro_f1=m["macro_f1"], dev_threshold=m["threshold"], dev_macro_f1_at_0_5=m["macro_f1_at_0_5"])
        history.append(record)
        message = (f"[{cfg['name']}] epoch {epoch + 1:2d}/{n_epochs} | train loss {record['train_loss']:.4f} | "
                   f"dev macro-F1 {m['macro_f1']:.4f} (threshold {m['threshold']:.2f}; at 0.50: {m['macro_f1_at_0_5']:.4f})")
        if m["macro_f1"] > best:
            best, bad = m["macro_f1"], 0
            save_atomic({"model": {k: v.detach().cpu().clone() for k, v in head.state_dict().items()}, "signature": signature,
                         "epoch": epoch + 1, "threshold": m["threshold"]}, model_path)
            message += "  <- best"
        else:
            bad += 1
        print(message, flush=True)
        if bad >= cfg["hier_patience"]:
            print(f"[{cfg['name']}] early stop: no improvement for {cfg['hier_patience']} epochs.")
            break
    if fixed_epochs:
        save_atomic({"model": {k: v.detach().cpu().clone() for k, v in head.state_dict().items()}, "signature": signature,
                     "epoch": n_epochs, "threshold": 0.5}, model_path)
    saved = torch_load(model_path)
    head.load_state_dict(saved["model"])
    info = {"best_step": saved["epoch"], "best_epoch": saved["epoch"], "history": history, "threshold": saved["threshold"],
            "train_seconds": time.time() - t_start, "peak_memory_mb": 0.0, "parameters": params, "train_cases": len(y_train)}
    return head, info

def load_hier_head(cfg, d, path):
    head = HierarchicalHead(d, cfg).to(DEVICE)
    head.load_state_dict(torch_load(path)["model"])
    return head

def stratified_folds(labels, k, seed):
    rng = np.random.RandomState(seed)
    fold = np.zeros(len(labels), dtype=np.int64)
    for c in (0, 1):
        idx = np.where(labels == c)[0]
        rng.shuffle(idx)
        fold[idx] = np.arange(len(idx)) % k
    return fold

def hierarchical_dev_cv(cfg, vectors, signature):
    """Stacking variant (H2): the chunk model was fine-tuned on train, so its vectors of TRAIN cases are better than
    on unseen cases. Here the head is trained only on cases the chunk model never saw - the dev set - with k-fold
    cross-validation: every dev case is scored by a head that did not train on it (honest dev score), and the test
    set is scored by the average of the k heads. -> (dev logits out-of-fold, test logits, info)."""
    folder = exp_dir(cfg["name"])
    y_dev = labels_of("dev")
    k = cfg["hier_folds"]
    fold = stratified_folds(y_dev, k, cfg["seed"])
    dev_logits = np.zeros((len(y_dev), 2), dtype=np.float32)
    test_logits = np.zeros((len(vectors["test"]), 2), dtype=np.float32)
    d = vectors["dev"][0].shape[1]
    t_start, infos = time.time(), []
    for f in range(k):
        path = os.path.join(folder, f"fold{f}.pt")
        tr, ho = np.where(fold != f)[0], np.where(fold == f)[0]
        if os.path.exists(path) and torch_load(path).get("signature") == signature:
            head = load_hier_head(cfg, d, path)
        else:
            head, info = train_hierarchical(cfg, [vectors["dev"][i] for i in tr], y_dev[tr], None, None, signature, path,
                                            fixed_epochs=cfg["hier_cv_epochs"], tag=f" fold {f + 1}/{k}")
            infos.append(info)
        dev_logits[ho] = hier_logits(head, [vectors["dev"][i] for i in ho])
        test_logits += hier_logits(head, vectors["test"]) / k
        m = binary_scores(y_dev[ho], probabilities(dev_logits[ho]) >= 0.5)
        print(f"[{cfg['name']}] fold {f + 1}/{k}: held-out macro-F1 at 0.50 = {m['macro_f1']:.4f} ({len(ho)} cases)")
        release(head)
    threshold, _ = best_threshold(y_dev, probabilities(dev_logits))
    params = count_parameters(HierarchicalHead(d, cfg))
    info = {"best_step": cfg["hier_cv_epochs"], "best_epoch": cfg["hier_cv_epochs"],
            "history": infos[0]["history"] if infos else [], "threshold": threshold, "train_seconds": time.time() - t_start,
            "peak_memory_mb": 0.0, "parameters": params, "train_cases": int(len(y_dev) * (k - 1) / k)}
    return dev_logits, test_logits, info

## 10. Run the experiments

For each experiment: train → dev score and threshold → test predictions → descriptive zip in `submissions/`. Finished experiments are loaded from disk. One failing experiment does not stop the others.

In [10]:
RESULTS = {}

def write_predictions(cfg, folder, predicted, dev_f1):
    """prediction.json with exactly one 0/1 decision per test case, zipped under a descriptive name inside the
    experiment folder AND in submissions/. Everything is checked before the file is written."""
    ids = [c["id"] for c in CASES["test"]]
    assert len(predicted) == len(ids)
    values = [int(v) for v in predicted]
    assert all(v in (0, 1) for v in values)
    if PREDICTION_FORMAT == "dict":
        output = {i: v for i, v in zip(ids, values)}
    else:
        output = [{"id": i, PREDICTION_KEY: v} for i, v in zip(ids, values)]
    prediction_path = os.path.join(folder, "prediction.json")
    with open(prediction_path, "w", encoding="utf-8") as f:
        json.dump(output, f, ensure_ascii=False)
    zip_name = submission_name(cfg, dev_f1)
    for old in glob.glob(os.path.join(folder, "phase4_*.zip")) + glob.glob(os.path.join(SUBMISSION_DIR, f"phase4_{cfg['name'].split('_')[0]}_*.zip")):
        os.remove(old)
    zip_path = os.path.join(folder, zip_name)
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(prediction_path, arcname="prediction.json")
    with zipfile.ZipFile(zip_path, "r") as z:
        assert z.namelist() == ["prediction.json"]
        check = json.loads(z.read("prediction.json").decode("utf-8"))
    if PREDICTION_FORMAT == "dict":
        assert set(check) == set(ids) and all(check[i] in (0, 1) for i in ids)
    else:
        assert [r["id"] for r in check] == ids and all(r[PREDICTION_KEY] in (0, 1) for r in check)
    shutil.copy2(zip_path, os.path.join(SUBMISSION_DIR, zip_name))
    return os.path.join(SUBMISSION_DIR, zip_name)

def teacher_logits_for(cfg):
    """Teacher's scores for every training case (its own tokenizer and section), cached in the student's folder."""
    teacher_cfg = experiment_config(cfg["teacher"])
    path = os.path.join(exp_dir(cfg["name"]), f"teacher_logits_{RESULTS[cfg['teacher']]['signature'][:10]}.npy")
    if os.path.exists(path):
        return np.load(path)
    teacher, _ = load_classifier(teacher_cfg)
    seqs = document_tokens(teacher_cfg["backbone"], teacher_cfg["section"], teacher_cfg["chunk_tokens"])["train"]
    logits = predict_logits(teacher, seqs, teacher_cfg["eval_batch_size"], desc=f"teacher {cfg['teacher']} scores")
    print(f"[{cfg['name']}] distilling from {cfg['teacher']} ({count_parameters(teacher)['total'] / 1e6:.1f}M parameters); "
          f"teacher accuracy on its own training cases: {(logits.argmax(1) == labels_of('train')).mean():.4f}")
    release(teacher)
    np.save(path, logits)
    return logits

def run_experiment(name):
    """Train (best evaluation on dev) -> threshold on dev -> test predictions -> submission zip.
    Every finished part is loaded from disk."""
    if name in RESULTS:
        return RESULTS[name]
    cfg = experiment_config(name)
    if cfg["backbone"] not in PHASE1:
        print(f"[{name}] SKIPPED: no Phase-1 checkpoint for model {cfg['backbone']} is attached.")
        return None
    depends_on = cfg["teacher"] or cfg["hier_base"]
    dependency_signature = ""
    if depends_on:
        dep = run_experiment(depends_on)
        if dep is None:
            print(f"[{name}] SKIPPED: {depends_on} (needed first) could not be run.")
            return None
        dependency_signature = dep["signature"]
    folder = exp_dir(name)
    signature = config_signature(cfg, dependency_signature)
    results_path = os.path.join(folder, "results.json")
    if os.path.exists(results_path):
        saved = load_json_file(results_path)
        if saved.get("signature") == signature and os.path.exists(saved.get("submission", "")):
            print(f"[{name}] already finished: dev macro-F1 {saved['dev']['macro_f1']:.4f} (loaded from disk)")
            RESULTS[name] = saved
            return saved

    model_path = os.path.join(folder, "model.pt")
    trained = os.path.exists(model_path) and torch_load(model_path).get("signature") == signature \
        and not os.path.exists(os.path.join(folder, "last.pt")) and os.path.exists(os.path.join(folder, "train_info.json"))
    if cfg["hier_base"]:
        base_cfg = experiment_config(cfg["hier_base"])
        vectors = hierarchical_vectors(cfg, base_cfg, dependency_signature)
        d = vectors["train"][0].shape[1]
        if cfg["hier_train_on"] == "dev_cv":
            cv_dev, cv_test, info = hierarchical_dev_cv(cfg, vectors, signature)
            model = None
            predict = lambda split: cv_dev if split == "dev" else cv_test
        else:
            if trained:
                model = load_hier_head(cfg, d, model_path)
                info = load_json_file(os.path.join(folder, "train_info.json"))
            else:
                model, info = train_hierarchical(cfg, vectors["train"], labels_of("train"), vectors["dev"], labels_of("dev"),
                                                 signature, model_path)
            predict = lambda split: hier_logits(model, vectors[split])
    else:
        if trained:
            model, _ = load_classifier(cfg)
            info = load_json_file(os.path.join(folder, "train_info.json"))
            print(f"[{name}] model already trained; loading it.")
        else:
            teacher = teacher_logits_for(cfg) if cfg["teacher"] else None
            model, info = train_classifier(cfg, signature, teacher)
        tokens = document_tokens(cfg["backbone"], cfg["section"], cfg["chunk_tokens"])
        predict = lambda split: predict_logits(model, tokens[split], cfg["eval_batch_size"], desc=f"{split} prediction")
    write_json(info, os.path.join(folder, "train_info.json"))

    y_dev = labels_of("dev")
    dev_prob = probabilities(predict("dev"))
    threshold = info["threshold"]
    dev_pred = (dev_prob >= threshold).astype(np.int64)
    dev = binary_scores(y_dev, dev_pred)
    dev.update(threshold=threshold, macro_f1_at_0_5=binary_scores(y_dev, dev_prob >= 0.5)["macro_f1"],
               by_length=scores_by_length(CASES["dev"], y_dev, dev_pred), best_epoch=info["best_epoch"],
               history=info["history"], parameters=info["parameters"], train_seconds=info["train_seconds"],
               peak_memory_mb=info["peak_memory_mb"])
    write_json({c["id"]: {"prob_accepted": float(p), "pred": int(y)} for c, p, y in zip(CASES["dev"], dev_prob, dev_pred)},
               os.path.join(folder, "dev_predictions.json"))
    print(f"[{name}] DEV macro-F1 {dev['macro_f1']:.4f} (threshold {threshold:.2f}; at 0.50: {dev['macro_f1_at_0_5']:.4f}) | "
          f"accuracy {dev['accuracy']:.4f} | F1 rejected {dev['class0']['f1']:.4f}, accepted {dev['class1']['f1']:.4f} | "
          f"best at epoch {info['best_epoch']}")

    t0 = time.time()
    test_prob = probabilities(predict("test"))
    inference_seconds = time.time() - t0
    test_pred = (test_prob >= threshold).astype(np.int64)
    zip_path = write_predictions(cfg, folder, test_pred, dev["macro_f1"])
    result = {"name": name, "signature": signature, "config": {k: v for k, v in cfg.items() if k != "name"},
              "phase1_checkpoint": PHASE1[cfg["backbone"]]["dir"], "dev": dev,
              "test_cases": len(test_pred), "test_predicted_accepted": int(test_pred.sum()),
              "inference_seconds": inference_seconds, "cases_per_second": len(test_pred) / max(1e-9, inference_seconds),
              "submission": zip_path}
    write_json(result, results_path)
    print(f"[{name}] test: {len(test_pred):,} cases, {int(test_pred.sum()):,} predicted accepted "
          f"({100 * test_pred.mean():.1f}%) -> {zip_path}\n", flush=True)
    release(model)
    RESULTS[name] = result
    return result

for _name in RUN_EXPERIMENTS:
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    try:
        run_experiment(_name)
    except Exception as _err:
        import traceback
        traceback.print_exc()
        print(f"[{_name}] FAILED: {_err} - continuing with the next experiment.\n")
        release()

Loaded Phase-1 model D (GPT decoder): 8/8 layers, 64.57M parameters
[R1_D_lorma] parameters: 64.97M total | trained: 0.393M of the 64.96M language-model weights + 0.0031M head (0.61% of all weights)


tokenising train (tail 2048) for model D:   0%|          | 0/37387 [00:00<?, ?it/s]

tokenising dev (tail 2048) for model D:   0%|          | 0/1988 [00:00<?, ?it/s]

tokenising test (tail 2048) for model D:   0%|          | 0/1517 [00:00<?, ?it/s]

tokenised tail 2048 tokens for model D: train cases shorter than that: 8,029 / 37,387


R1_D_lorma epoch 1/3:   0%|          | 0/2337 [00:00<?, ?it/s]

... (215 outputs left out)

dev prediction:   0%|          | 0/32 [00:00<?, ?it/s]

[R3_C_bitfit] DEV macro-F1 0.5694 (threshold 0.48; at 0.50: 0.5471) | accuracy 0.5694 | F1 rejected 0.5711, accepted 0.5677 | best at epoch 1.0


test prediction:   0%|          | 0/24 [00:00<?, ?it/s]

[R3_C_bitfit] test: 1,517 cases, 733 predicted accepted (48.3%) -> /kaggle/working/cs781_phase4/submissions/phase4_R3_C-EncDec_BitFit_last512_dev0.5694.zip



## 11. Results and analysis

In [11]:
finished = {name: r for name, r in RESULTS.items() if r is not None}
for _name in EXPERIMENTS:
    _path = os.path.join(WORK_ROOT, _name, "results.json")
    if _name not in finished and os.path.exists(_path):
        finished[_name] = load_json_file(_path)

rows = []
for name, r in sorted(finished.items(), key=lambda kv: -kv[1]["dev"]["macro_f1"]):
    cfg, dev = r["config"], r["dev"]
    part = (f"last {cfg['hier_chunks']}x{cfg['chunk_tokens']}" if cfg["hier_base"]
            else ("last " if cfg["section"] == "tail" else "first ") + str(cfg["chunk_tokens"]))
    technique = ("hier on " + cfg["hier_base"].split("_")[0] + (" cv" if cfg.get("hier_train_on") == "dev_cv" else "")) if cfg["hier_base"] else \
        cfg["technique"] + (" + KD" if cfg["teacher"] else "")
    rows.append({"experiment": name, "model": cfg["backbone"] + (f"[:{cfg['n_layers']}]" if cfg["n_layers"] else ""),
                 "technique": technique, "reads": part,
                 "dev_macro_f1": dev["macro_f1"], "dev_f1_at_0_5": dev["macro_f1_at_0_5"], "threshold": dev["threshold"],
                 "dev_accuracy": dev["accuracy"], "f1_rejected": dev["class0"]["f1"], "f1_accepted": dev["class1"]["f1"],
                 "best_epoch": dev["best_epoch"], "params_M": dev["parameters"]["total"] / 1e6,
                 "trainable_M": dev["parameters"]["trainable"] / 1e6, "trainable_%": dev["parameters"]["trainable_percent"],
                 "train_min": dev["train_seconds"] / 60, "peak_mem_GB": dev["peak_memory_mb"] / 1000,
                 "test_cases_per_s": r["cases_per_second"], "test_accepted_%": 100 * r["test_predicted_accepted"] / r["test_cases"],
                 "submission": os.path.basename(r["submission"])})

if not rows:
    print("No finished experiments yet.")
else:
    header = (f"{'experiment':22s} {'model':6s} {'technique':14s} {'reads':13s} {'macroF1':>8s} {'@0.5':>6s} {'thr':>5s} "
              f"{'acc':>6s} {'F1 rej':>6s} {'F1 acc':>6s} {'params':>7s} {'trained':>8s} {'%':>6s} {'train':>6s} {'mem':>5s} {'case/s':>7s}")
    print(f"Dev set ({len(CASES['dev']):,} cases). macroF1 = mean F1 of the two classes at the dev-tuned threshold 'thr'; @0.5 = at threshold 0.5.\n"
          + header + "\n" + "-" * len(header))
    for x in rows:
        print(f"{x['experiment']:22s} {x['model']:6s} {x['technique']:14s} {x['reads']:13s} {x['dev_macro_f1']:8.4f} "
              f"{x['dev_f1_at_0_5']:6.4f} {x['threshold']:5.2f} {x['dev_accuracy']:6.4f} {x['f1_rejected']:6.4f} {x['f1_accepted']:6.4f} "
              f"{x['params_M']:6.1f}M {x['trainable_M']:7.3f}M {x['trainable_%']:6.2f} {x['train_min']:5.1f}m "
              f"{x['peak_mem_GB']:4.1f}G {x['test_cases_per_s']:7.0f}")
    with open(os.path.join(WORK_ROOT, "results_summary.csv"), "w", encoding="utf-8") as f:
        f.write(",".join(rows[0].keys()) + "\n")
        for x in rows:
            f.write(",".join(str(round(v, 5)) if isinstance(v, float) else str(v) for v in x.values()) + "\n")

    print("\nDev macro-F1 by document length (does reading only one part fail on long judgments?):")
    buckets = [lab for _, _, lab in LENGTH_BUCKETS]
    print(f"{'experiment':22s} " + " ".join(f"{b:>14s}" for b in buckets))
    for x in rows:
        by = finished[x["experiment"]]["dev"]["by_length"]
        print(f"{x['experiment']:22s} " + " ".join(f"{by[b]['macro_f1']:8.4f} ({by[b]['cases']:4d})" if b in by else f"{'-':>14s}"
                                                    for b in buckets))
    print("\nConfusion matrices on dev (rows = gold rejected / accepted, columns = predicted):")
    for x in rows:
        (a, b), (c, d) = finished[x["experiment"]]["dev"]["confusion"]
        print(f"  {x['experiment']:22s} [[{a:5d} {b:5d}]  [{c:5d} {d:5d}]]")
    print(f"\nSummary table saved to {os.path.join(WORK_ROOT, 'results_summary.csv')}")

Dev set (1,988 cases). macroF1 = mean F1 of the two classes at the dev-tuned threshold 'thr'; @0.5 = at threshold 0.5.
experiment             model  technique      reads          macroF1   @0.5   thr    acc F1 rej F1 acc  params  trained      %  train   mem  case/s
--------------------------------------------------------------------------------------------------------------------------------------------------
H4_B_hier_devcv        B      hier on R2b cv last 4x512      0.7344 0.7344  0.50 0.7344 0.7317 0.7371    1.0M   0.986M 100.00   0.2m  0.0G 10516957
H5_D_hier_devcv        D      hier on R5 cv  last 4x512      0.6958 0.6958  0.50 0.6962 0.7062 0.6854    1.0M   0.986M 100.00   0.2m  0.0G 9384601
R2b_B_adapter_6ep      B      adapter        last 512        0.6801 0.6666  0.52 0.6801 0.6794 0.6807   67.7M   1.602M   2.37  42.9m  3.5G     141
R5_D_adapter           D      adapter        last 512        0.6670 0.6116  0.32 0.6670 0.6660 0.6680   66.2M   1.615M   2.44  42.9m  2.3G     18

## 12. Submission files

In [12]:
if rows:
    print(f"Submission files in {SUBMISSION_DIR} (best dev score first):")
    for x in rows:
        print(f"  dev macro-F1 {x['dev_macro_f1']:.4f}   {x['submission']}")
    print(f"\nprediction.json format: {PREDICTION_FORMAT!r}; first entries of {rows[0]['submission']}:")
    with zipfile.ZipFile(os.path.join(SUBMISSION_DIR, rows[0]["submission"])) as z:
        _p = json.loads(z.read("prediction.json"))
    print(" ", (_p[:3] if isinstance(_p, list) else dict(list(_p.items())[:3])), f"... ({len(_p)} cases)")

Submission files in /kaggle/working/cs781_phase4/submissions (best dev score first):
  dev macro-F1 0.7344   phase4_H4_B-GQA-RoPE_HierBiLSTM-on-R2b-devCV_last4x512_dev0.7344.zip
  dev macro-F1 0.6958   phase4_H5_D-GPT400M_HierBiLSTM-on-R5-devCV_last4x512_dev0.6958.zip
  dev macro-F1 0.6801   phase4_R2b_B-GQA-RoPE_Adapter_last512_dev0.6801.zip
  dev macro-F1 0.6670   phase4_R5_D-GPT400M_Adapter_last512_dev0.6670.zip
  dev macro-F1 0.6670   phase4_R4_A-GPT220M-first4layers_KD-from-R1_last512_dev0.6670.zip
  dev macro-F1 0.6307   phase4_R1_D-GPT400M_LoRMA_last512_dev0.6307.zip
  dev macro-F1 0.5694   phase4_R3_C-EncDec_BitFit_last512_dev0.5694.zip

prediction.json format: 'list'; first entries of phase4_H4_B-GQA-RoPE_HierBiLSTM-on-R2b-devCV_last4x512_dev0.7344.zip:
  [{'id': '1986_397', 'pred': 1}, {'id': '1977_183', 'pred': 0}, {'id': '1993_98', 'pred': 1}] ... (1517 cases)
